# Phase 3D — Complete Fixed Colab Notebook

Use **CPU / High-RAM** if available. Do not use TPU for Phase 3D: MOFA+ and SCOT here run through NumPy/SciPy/POT CPU code.

This notebook executes the frozen 30-run Phase 3D benchmark:

- A1, D1, E11, E13, E15
- MOFA+ and SCOT
- seeds 1729, 2718, 31415
- 30 runs total
- E18 excluded because ATAC is not verified

Fixes included: public GitHub clone, exact Python 3.11.8 subprocess environment, exact frozen packages, folder-level `gdown`, automatic nested-folder flattening, dataset SHA-256 checks, config SHA-256 checks, frozen SCOT commit, in-repo temporary configs, NumPy-safe JSON serialization, deferred registry mutation, resumable Google Drive outputs, partial-output preservation, and final aggregation.

In [ ]:
# 1. Mount Google Drive and define the study

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import sys
import json
import csv
import math
import time
import shutil
import hashlib
import platform
import subprocess
from collections import defaultdict

REPO_URL = "https://github.com/rifatahsanpul0k/research.git"
REPO_ROOT = Path("/content/multiomics-research")

DRIVE_BASE = Path("/content/drive/MyDrive/phase3d_results")
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

DATASET_SOURCES = {
    "10x_human_lymph_node_A1":
        "https://drive.google.com/drive/folders/10z1N4MwW8Y49o8GlkYGBKVx1N7fiMuyC",
    "10x_human_lymph_node_D1":
        "https://drive.google.com/drive/folders/1-g_Ca2XMaMXF-MisuVY-wobWDX86O6zz",
    "Mouse_Brain_E11_S1":
        "https://drive.google.com/drive/folders/1zRwDJrYnks0LRzlAVRqPU7jE_OcStgPo",
    "Mouse_Brain_E13_S1":
        "https://drive.google.com/drive/folders/1GOufwIRjjfcd9Bi2GKtebzKoPCg2jVud",
    "Mouse_Brain_E15_S1":
        "https://drive.google.com/drive/folders/1rHkTL5OF5qPsEERypRGMS51SjUQ69tdD",
}

REQUIRED_FILES = {
    "10x_human_lymph_node_A1":
        ["adata_RNA.h5ad", "adata_ADT.h5ad", "annotation.csv"],
    "10x_human_lymph_node_D1":
        ["adata_RNA.h5ad", "adata_ADT.h5ad", "annotation.csv"],
    "Mouse_Brain_E11_S1":
        ["adata_RNA.h5ad", "adata_ATAC.h5ad", "anno.csv"],
    "Mouse_Brain_E13_S1":
        ["adata_RNA.h5ad", "adata_ATAC.h5ad", "anno.csv"],
    "Mouse_Brain_E15_S1":
        ["adata_RNA.h5ad", "adata_ATAC.h5ad", "anno.csv"],
}

EXPECTED_CONFIGS = [
    "EXP-LN-A1-MOFAPLUS-KMEANS-S1729.json",
    "EXP-LN-A1-MOFAPLUS-KMEANS-S2718.json",
    "EXP-LN-A1-MOFAPLUS-KMEANS-S31415.json",
    "EXP-LN-A1-SCOT-KMEANS-S1729.json",
    "EXP-LN-A1-SCOT-KMEANS-S2718.json",
    "EXP-LN-A1-SCOT-KMEANS-S31415.json",
    "EXP-LN-D1-MOFAPLUS-KMEANS-S1729.json",
    "EXP-LN-D1-MOFAPLUS-KMEANS-S2718.json",
    "EXP-LN-D1-MOFAPLUS-KMEANS-S31415.json",
    "EXP-LN-D1-SCOT-KMEANS-S1729.json",
    "EXP-LN-D1-SCOT-KMEANS-S2718.json",
    "EXP-LN-D1-SCOT-KMEANS-S31415.json",
    "EXP-MB-E11-MOFAPLUS-KMEANS-S1729.json",
    "EXP-MB-E11-MOFAPLUS-KMEANS-S2718.json",
    "EXP-MB-E11-MOFAPLUS-KMEANS-S31415.json",
    "EXP-MB-E11-SCOT-KMEANS-S1729.json",
    "EXP-MB-E11-SCOT-KMEANS-S2718.json",
    "EXP-MB-E11-SCOT-KMEANS-S31415.json",
    "EXP-MB-E13-MOFAPLUS-KMEANS-S1729.json",
    "EXP-MB-E13-MOFAPLUS-KMEANS-S2718.json",
    "EXP-MB-E13-MOFAPLUS-KMEANS-S31415.json",
    "EXP-MB-E13-SCOT-KMEANS-S1729.json",
    "EXP-MB-E13-SCOT-KMEANS-S2718.json",
    "EXP-MB-E13-SCOT-KMEANS-S31415.json",
    "EXP-MB-E15-MOFAPLUS-KMEANS-S1729.json",
    "EXP-MB-E15-MOFAPLUS-KMEANS-S2718.json",
    "EXP-MB-E15-MOFAPLUS-KMEANS-S31415.json",
    "EXP-MB-E15-SCOT-KMEANS-S1729.json",
    "EXP-MB-E15-SCOT-KMEANS-S2718.json",
    "EXP-MB-E15-SCOT-KMEANS-S31415.json",
]

print("Colab host Python:", platform.python_version())
print("Experiments:", len(EXPECTED_CONFIGS))


In [ ]:
# 2. Clone the public repository

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

subprocess.run(
    ["git", "clone", REPO_URL, str(REPO_ROOT)],
    check=True,
)

HEAD = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()

BRANCH = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "--abbrev-ref", "HEAD"],
    text=True,
).strip()

DRIVE_OUT = DRIVE_BASE / HEAD[:12]
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

print("Branch:", BRANCH)
print("HEAD:", HEAD)
print("Persistent results:", DRIVE_OUT)


In [ ]:
# 3. Build exact Python 3.11.8 experiment environment

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "uv", "gdown"],
    check=True,
)

UV = shutil.which("uv")
if UV is None:
    raise RuntimeError("uv executable not found")

VENV = Path("/content/phase3d-py311")
if VENV.exists():
    shutil.rmtree(VENV)

subprocess.run(
    [UV, "venv", str(VENV), "--python", "3.11.8", "--seed"],
    check=True,
)

PY311 = VENV / "bin" / "python"

FROZEN_PACKAGES = [
    "numpy==2.4.4",
    "scipy==1.17.1",
    "scikit-learn==1.9.1",
    "h5py==3.16.0",
    "mofapy2==0.7.5",
    "POT==0.9.6.post1",
]

subprocess.run(
    [UV, "pip", "install", "--python", str(PY311), *FROZEN_PACKAGES],
    check=True,
)

verify_code = """
import platform
from importlib.metadata import version

expected = {
    "numpy": "2.4.4",
    "scipy": "1.17.1",
    "scikit-learn": "1.9.1",
    "h5py": "3.16.0",
    "mofapy2": "0.7.5",
    "POT": "0.9.6.post1",
}

print("Python", platform.python_version())
assert platform.python_version() == "3.11.8"

for package, wanted in expected.items():
    actual = version(package)
    print(package, actual)
    assert actual == wanted, (package, actual, wanted)
"""

subprocess.run([str(PY311), "-c", verify_code], check=True)
print("Exact Phase 3D environment verified.")


In [ ]:
# 4. Verify the 30 frozen configs

CONFIG_DIR = REPO_ROOT / "07_models" / "02_classical_integration" / "configs"
CHECKSUM_FILE = REPO_ROOT / "07_models" / "02_classical_integration" / "CONFIG_CHECKSUMS.json"
RUNNER_SOURCE = REPO_ROOT / "code" / "benchmarks" / "run_phase3d.py"

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

config_checksums = json.loads(CHECKSUM_FILE.read_text())

for name in EXPECTED_CONFIGS:
    path = CONFIG_DIR / name
    if not path.exists():
        raise FileNotFoundError(path)

    rel = str(path.relative_to(REPO_ROOT))
    expected = config_checksums.get(rel)
    actual = sha256_file(path)

    if expected is None:
        raise RuntimeError(f"Missing config checksum entry: {rel}")
    if actual != expected:
        raise RuntimeError(
            f"Config checksum mismatch:\n{rel}\nexpected={expected}\nactual={actual}"
        )

print("Verified all 30 frozen config checksums.")


In [ ]:
# 5. Download datasets with folder-level gdown, flatten nesting, verify SHA-256

MANIFEST_PATH = (
    REPO_ROOT
    / "02_omics"
    / "01_measurement_and_data_generation"
    / "DOWNLOAD_MANIFEST.json"
)

manifest = json.loads(MANIFEST_PATH.read_text())

manifest_hash = {
    (record["dataset_id"], record["filename"]): record["sha256"]
    for record in manifest
    if record.get("status") == "complete"
}

STAGING_ROOT = Path("/content/phase3d_dataset_downloads")
STAGING_ROOT.mkdir(parents=True, exist_ok=True)

def verified_target(dataset: str, filename: str) -> bool:
    path = REPO_ROOT / "04_datasets" / dataset / "raw" / filename
    expected = manifest_hash.get((dataset, filename))
    return path.exists() and expected is not None and sha256_file(path) == expected

for dataset, folder_url in DATASET_SOURCES.items():
    required = REQUIRED_FILES[dataset]

    print("\n" + "=" * 100)
    print("DATASET:", dataset)
    print("=" * 100)

    if all(verified_target(dataset, filename) for filename in required):
        print("Already checksum-verified; skipping download.")
        continue

    staging = STAGING_ROOT / dataset
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir(parents=True)

    cmd = [
        sys.executable,
        "-m",
        "gdown",
        "--folder",
        folder_url,
        "--output",
        str(staging),
    ]

    result = subprocess.run(
        cmd,
        text=True,
        capture_output=True,
        check=False,
    )

    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            f"gdown failed for {dataset}; return code={result.returncode}"
        )

    raw_dir = REPO_ROOT / "04_datasets" / dataset / "raw"
    raw_dir.mkdir(parents=True, exist_ok=True)

    for filename in required:
        expected = manifest_hash.get((dataset, filename))
        if expected is None:
            raise RuntimeError(f"No manifest checksum for {dataset}/{filename}")

        matches = [p for p in staging.rglob(filename) if p.is_file()]

        if len(matches) == 0:
            raise FileNotFoundError(
                f"{filename} not found recursively under {staging}"
            )
        if len(matches) > 1:
            raise RuntimeError(
                "Multiple matches:\n" + "\n".join(str(p) for p in matches)
            )

        source = matches[0]
        target = raw_dir / filename
        shutil.copy2(source, target)

        actual = sha256_file(target)
        if actual != expected:
            raise RuntimeError(
                f"Dataset checksum mismatch:\n"
                f"{dataset}/{filename}\nexpected={expected}\nactual={actual}"
            )

        size_mb = target.stat().st_size / (1024 ** 2)
        print(f"✓ {filename:<25} {size_mb:,.2f} MB")

print("\nAll Phase 3D raw files downloaded, flattened, and verified.")


In [ ]:
# 6. Dataset preflight

for dataset, required in REQUIRED_FILES.items():
    raw_dir = REPO_ROOT / "04_datasets" / dataset / "raw"
    print("\n" + dataset)

    for filename in required:
        path = raw_dir / filename
        expected = manifest_hash[(dataset, filename)]

        if not path.exists():
            raise FileNotFoundError(path)
        if sha256_file(path) != expected:
            raise RuntimeError(f"Checksum mismatch: {path}")

        print(f"✓ {filename}")

print("\nDATASET PREFLIGHT PASSED")


In [ ]:
# 7. Verify official SCOT source

SCOT_ROOT = REPO_ROOT / "external" / "SCOT"
SCOT_COMMIT = "14649be6e14017dcfe7ba619091b33d1df55f6a9"

if SCOT_ROOT.exists() and not (SCOT_ROOT / ".git").exists():
    shutil.rmtree(SCOT_ROOT)

if not SCOT_ROOT.exists():
    SCOT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "https://github.com/rsinghlab/SCOT.git", str(SCOT_ROOT)],
        check=True,
    )

subprocess.run(
    ["git", "-C", str(SCOT_ROOT), "fetch", "--all", "--tags"],
    check=True,
)

subprocess.run(
    ["git", "-C", str(SCOT_ROOT), "checkout", "--detach", SCOT_COMMIT],
    check=True,
)

actual_scot = subprocess.check_output(
    ["git", "-C", str(SCOT_ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()

if actual_scot != SCOT_COMMIT:
    raise RuntimeError(f"SCOT commit mismatch: {actual_scot}")

print("SCOT verified:", actual_scot)


In [ ]:
# 8. Build a runtime copy of the repository runner

RUNTIME_RUNNER = (
    REPO_ROOT
    / "code"
    / "benchmarks"
    / "run_phase3d_colab_runtime.py"
)

source = RUNNER_SOURCE.read_text(encoding="utf-8")

old_import = "from code.benchmarks.common.io import write_json_atomic"
new_import = (
    "from code.benchmarks.common.io "
    "import write_json_atomic as _base_write_json_atomic"
)

if old_import not in source:
    raise RuntimeError("write_json_atomic import anchor not found")

source = source.replace(old_import, new_import, 1)

root_marker = 'ROOT = Path(__file__).resolve().parents[2]'

json_patch = r'''
def _json_ready(value):
    import numpy as _np

    if isinstance(value, _np.ndarray):
        return value.tolist()
    if isinstance(value, _np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(k): _json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_ready(v) for v in value]
    return value


def write_json_atomic(value, path):
    return _base_write_json_atomic(_json_ready(value), path)


'''

if root_marker not in source:
    raise RuntimeError("ROOT anchor not found")

source = source.replace(root_marker, json_patch + root_marker, 1)

registry_start = source.find(
    "def append_registry(row: dict[str, object]) -> None:"
)
run_start = source.find(
    "\ndef run(config_path: Path)",
    registry_start,
)

if registry_start == -1 or run_start == -1:
    raise RuntimeError("append_registry anchor not found")

registry_stub = (
    "def append_registry(row: dict[str, object]) -> None:\n"
    "    # Registry reconciliation is deferred until post-run review.\n"
    "    return None\n\n"
)

source = (
    source[:registry_start]
    + registry_stub
    + source[run_start + 1:]
)

old_main = 'print(json.dumps(run(args.config), indent=2, sort_keys=True))'
new_main = (
    'print(json.dumps(_json_ready(run(args.config)), '
    'indent=2, sort_keys=True))'
)

if old_main not in source:
    raise RuntimeError("CLI JSON anchor not found")

source = source.replace(old_main, new_main, 1)

RUNTIME_RUNNER.write_text(source, encoding="utf-8")

subprocess.run(
    [str(PY311), "-m", "py_compile", str(RUNTIME_RUNNER)],
    cwd=REPO_ROOT,
    check=True,
)

print("Runtime runner prepared:", RUNTIME_RUNNER)
print("Runtime runner compiles under Python 3.11.8")


In [ ]:
# 9. Prepare temporary execution configs INSIDE the repository

TEMP_CONFIG_DIR = REPO_ROOT / ".phase3d_colab_configs"
TEMP_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

def prepare_temp_config(name: str) -> Path:
    source_path = CONFIG_DIR / name
    original = json.loads(source_path.read_text())

    runtime_cfg = json.loads(json.dumps(original))
    source_start_commit = runtime_cfg.get("start_commit")

    runtime_cfg["start_commit"] = HEAD
    runtime_cfg["manual_remote_execution"] = {
        "mode": "COLAB_MANUAL_PHASE3D",
        "source_config": str(source_path.relative_to(REPO_ROOT)),
        "source_config_sha256": sha256_file(source_path),
        "source_start_commit": source_start_commit,
        "execution_git_head": HEAD,
        "registry_write": "DEFERRED_UNTIL_POST_RUN_REVIEW",
    }

    target = TEMP_CONFIG_DIR / name
    target.write_text(
        json.dumps(runtime_cfg, indent=2, sort_keys=True) + "\n"
    )

    original_guard = json.loads(json.dumps(original))
    runtime_guard = json.loads(target.read_text())

    original_guard.pop("start_commit", None)
    runtime_guard.pop("start_commit", None)
    runtime_guard.pop("manual_remote_execution", None)

    if original_guard != runtime_guard:
        raise RuntimeError(
            f"Scientific config fields changed unexpectedly: {name}"
        )

    return target

for name in EXPECTED_CONFIGS:
    prepare_temp_config(name)

print("Prepared 30 runtime configs:", TEMP_CONFIG_DIR)
print("Scientific fields preserved.")


In [ ]:
# 10. Define robust execution/resume helpers

REQUIRED_SUCCESS_ARTIFACTS = [
    "STATUS",
    "config.json",
    "preprocessing.json",
    "identity_map.csv",
    "embedding.npz",
    "clusters.csv",
    "metrics.json",
    "validation.json",
    "provenance.json",
    "stdout.log",
    "stderr.log",
    "saved_artifacts_manifest.json",
]

def complete_output(path: Path) -> bool:
    if not path.exists():
        return False

    if not all((path / name).exists() for name in REQUIRED_SUCCESS_ARTIFACTS):
        return False

    if (path / "STATUS").read_text().strip() != "SUCCEEDED":
        return False

    for name in [
        "config.json",
        "preprocessing.json",
        "metrics.json",
        "validation.json",
        "provenance.json",
        "saved_artifacts_manifest.json",
    ]:
        json.loads((path / name).read_text())

    return True


def preserve_directory(path: Path, destination_root: Path, label: str) -> Path:
    destination_root.mkdir(parents=True, exist_ok=True)
    stamp = time.strftime("%Y%m%d_%H%M%S")
    target = destination_root / f"{path.name}__{label}__{stamp}"

    counter = 1
    while target.exists():
        target = destination_root / (
            f"{path.name}__{label}__{stamp}_{counter}"
        )
        counter += 1

    path.rename(target)
    return target


def run_one(config_name: str) -> str:
    if config_name not in EXPECTED_CONFIGS:
        raise ValueError(config_name)

    temp_config = TEMP_CONFIG_DIR / config_name
    if not temp_config.exists():
        temp_config = prepare_temp_config(config_name)

    cfg = json.loads(temp_config.read_text())
    exp_id = cfg["experiment_id"]

    runtime_output = REPO_ROOT / "08_experiments" / exp_id
    drive_output = DRIVE_OUT / exp_id

    print("\n" + "=" * 110)
    print("EXPERIMENT:", exp_id)
    print("DATASET:", cfg["dataset"])
    print("METHOD:", cfg["method"])
    print("SEED:", cfg["seed"])
    print("=" * 110)

    if drive_output.exists():
        if complete_output(drive_output):
            print("Already fully SUCCEEDED in Google Drive — skipping.")
            return "SUCCEEDED"

        preserved = preserve_directory(
            drive_output,
            DRIVE_OUT / "_partial_or_failed",
            "PARTIAL_DRIVE",
        )
        print("Preserved incomplete Drive output:", preserved)

    if runtime_output.exists():
        if complete_output(runtime_output):
            shutil.copytree(runtime_output, drive_output)
            print("Complete runtime output copied to Drive.")
            return "SUCCEEDED"

        preserved = preserve_directory(
            runtime_output,
            REPO_ROOT
            / "08_experiments"
            / "_partial_or_failed_colab",
            "PREVIOUS_RUNTIME",
        )
        print("Preserved prior runtime output:", preserved)

    logs_dir = DRIVE_OUT / "_logs"
    logs_dir.mkdir(parents=True, exist_ok=True)
    log_path = logs_dir / f"{exp_id}.combined.log"

    env = os.environ.copy()
    env["ASTRA_COLAB_CONFIRMED"] = "1"
    env["COLAB_RUNTIME_TYPE"] = "HIGH_RAM_CPU"
    env["ASTRA_COLAB_RAM_CLASS"] = "COLAB_RUNTIME_RECORDED"
    env["PYTHONPATH"] = str(REPO_ROOT)

    env["OMP_NUM_THREADS"] = "1"
    env["OPENBLAS_NUM_THREADS"] = "1"
    env["MKL_NUM_THREADS"] = "1"
    env["VECLIB_MAXIMUM_THREADS"] = "1"
    env["LOKY_MAX_CPU_COUNT"] = "1"

    command = [
        str(PY311),
        str(RUNTIME_RUNNER),
        "--config",
        str(temp_config),
    ]

    print("Python:", PY311)
    print("Config:", temp_config)
    print("Persistent output:", drive_output)
    print("Starting...")

    started = time.perf_counter()

    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            command,
            cwd=REPO_ROOT,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        assert process.stdout is not None

        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line, end="")

        return_code = process.wait()

    elapsed = time.perf_counter() - started

    print("\nReturn code:", return_code)
    print("Controller elapsed seconds:", round(elapsed, 2))
    print("Combined log:", log_path)

    if return_code != 0:
        if runtime_output.exists():
            failed_root = DRIVE_OUT / "_failed_runs"
            failed_root.mkdir(parents=True, exist_ok=True)

            stamp = time.strftime("%Y%m%d_%H%M%S")
            failed_copy = failed_root / f"{exp_id}__FAILED__{stamp}"
            shutil.copytree(runtime_output, failed_copy)

            print("Failed artifacts preserved:", failed_copy)

        print("FAILED:", exp_id)
        return "FAILED"

    if not complete_output(runtime_output):
        raise RuntimeError(
            f"{exp_id} returned 0 but the success artifact set is incomplete"
        )

    if drive_output.exists():
        shutil.rmtree(drive_output)

    shutil.copytree(runtime_output, drive_output)

    if not complete_output(drive_output):
        raise RuntimeError(
            f"Drive copy verification failed: {exp_id}"
        )

    metrics = json.loads((drive_output / "metrics.json").read_text())

    def metric_value(key):
        value = metrics.get(key)
        return value.get("result") if isinstance(value, dict) else value

    print("\nSUCCEEDED:", exp_id)
    print("ARI:", metric_value("ARI"))
    print("NMI:", metric_value("NMI"))
    print("Silhouette:", metric_value("silhouette"))

    return "SUCCEEDED"

print("run_one() ready.")


## Experiments 1–30

These are separate on purpose. You can use **Runtime → Run all**; they still execute sequentially. If Colab disconnects, rerun from the top. Completed runs stored in Drive are detected and skipped.

In [ ]:
# Experiment 01/30
status_01 = run_one("EXP-LN-A1-MOFAPLUS-KMEANS-S1729.json")
print("Experiment 01 status:", status_01)


In [ ]:
# Experiment 02/30
status_02 = run_one("EXP-LN-A1-MOFAPLUS-KMEANS-S2718.json")
print("Experiment 02 status:", status_02)


In [ ]:
# Experiment 03/30
status_03 = run_one("EXP-LN-A1-MOFAPLUS-KMEANS-S31415.json")
print("Experiment 03 status:", status_03)


In [ ]:
# Experiment 04/30
status_04 = run_one("EXP-LN-A1-SCOT-KMEANS-S1729.json")
print("Experiment 04 status:", status_04)


In [ ]:
# Experiment 05/30
status_05 = run_one("EXP-LN-A1-SCOT-KMEANS-S2718.json")
print("Experiment 05 status:", status_05)


In [ ]:
# Experiment 06/30
status_06 = run_one("EXP-LN-A1-SCOT-KMEANS-S31415.json")
print("Experiment 06 status:", status_06)


In [ ]:
# Experiment 07/30
status_07 = run_one("EXP-LN-D1-MOFAPLUS-KMEANS-S1729.json")
print("Experiment 07 status:", status_07)


In [ ]:
# Experiment 08/30
status_08 = run_one("EXP-LN-D1-MOFAPLUS-KMEANS-S2718.json")
print("Experiment 08 status:", status_08)


In [ ]:
# Experiment 09/30
status_09 = run_one("EXP-LN-D1-MOFAPLUS-KMEANS-S31415.json")
print("Experiment 09 status:", status_09)


In [ ]:
# Experiment 10/30
status_10 = run_one("EXP-LN-D1-SCOT-KMEANS-S1729.json")
print("Experiment 10 status:", status_10)


In [ ]:
# Experiment 11/30
status_11 = run_one("EXP-LN-D1-SCOT-KMEANS-S2718.json")
print("Experiment 11 status:", status_11)


In [ ]:
# Experiment 12/30
status_12 = run_one("EXP-LN-D1-SCOT-KMEANS-S31415.json")
print("Experiment 12 status:", status_12)


In [ ]:
# Experiment 13/30
status_13 = run_one("EXP-MB-E11-MOFAPLUS-KMEANS-S1729.json")
print("Experiment 13 status:", status_13)


In [ ]:
# Experiment 14/30
status_14 = run_one("EXP-MB-E11-MOFAPLUS-KMEANS-S2718.json")
print("Experiment 14 status:", status_14)


In [ ]:
# Experiment 15/30
status_15 = run_one("EXP-MB-E11-MOFAPLUS-KMEANS-S31415.json")
print("Experiment 15 status:", status_15)


In [ ]:
# Experiment 16/30
status_16 = run_one("EXP-MB-E11-SCOT-KMEANS-S1729.json")
print("Experiment 16 status:", status_16)


In [ ]:
# Experiment 17/30
status_17 = run_one("EXP-MB-E11-SCOT-KMEANS-S2718.json")
print("Experiment 17 status:", status_17)


In [ ]:
# Experiment 18/30
status_18 = run_one("EXP-MB-E11-SCOT-KMEANS-S31415.json")
print("Experiment 18 status:", status_18)


In [ ]:
# Experiment 19/30
status_19 = run_one("EXP-MB-E13-MOFAPLUS-KMEANS-S1729.json")
print("Experiment 19 status:", status_19)


In [ ]:
# Experiment 20/30
status_20 = run_one("EXP-MB-E13-MOFAPLUS-KMEANS-S2718.json")
print("Experiment 20 status:", status_20)


In [ ]:
# Experiment 21/30
status_21 = run_one("EXP-MB-E13-MOFAPLUS-KMEANS-S31415.json")
print("Experiment 21 status:", status_21)


In [ ]:
# Experiment 22/30
status_22 = run_one("EXP-MB-E13-SCOT-KMEANS-S1729.json")
print("Experiment 22 status:", status_22)


In [ ]:
# Experiment 23/30
status_23 = run_one("EXP-MB-E13-SCOT-KMEANS-S2718.json")
print("Experiment 23 status:", status_23)


In [ ]:
# Experiment 24/30
status_24 = run_one("EXP-MB-E13-SCOT-KMEANS-S31415.json")
print("Experiment 24 status:", status_24)


In [ ]:
# Experiment 25/30
status_25 = run_one("EXP-MB-E15-MOFAPLUS-KMEANS-S1729.json")
print("Experiment 25 status:", status_25)


In [ ]:
# Experiment 26/30
status_26 = run_one("EXP-MB-E15-MOFAPLUS-KMEANS-S2718.json")
print("Experiment 26 status:", status_26)


In [ ]:
# Experiment 27/30
status_27 = run_one("EXP-MB-E15-MOFAPLUS-KMEANS-S31415.json")
print("Experiment 27 status:", status_27)


In [ ]:
# Experiment 28/30
status_28 = run_one("EXP-MB-E15-SCOT-KMEANS-S1729.json")
print("Experiment 28 status:", status_28)


In [ ]:
# Experiment 29/30
status_29 = run_one("EXP-MB-E15-SCOT-KMEANS-S2718.json")
print("Experiment 29 status:", status_29)


In [ ]:
# Experiment 30/30
status_30 = run_one("EXP-MB-E15-SCOT-KMEANS-S31415.json")
print("Experiment 30 status:", status_30)


In [ ]:
# Final aggregation and completion gate

def metric_value(metrics, key):
    value = metrics.get(key)
    return value.get("result") if isinstance(value, dict) else value

rows = []

for config_name in EXPECTED_CONFIGS:
    source_cfg = json.loads((CONFIG_DIR / config_name).read_text())
    exp_id = source_cfg["experiment_id"]
    exp_dir = DRIVE_OUT / exp_id

    status = "NOT_RUN"
    if (exp_dir / "STATUS").exists():
        status = (exp_dir / "STATUS").read_text().strip()

    complete = complete_output(exp_dir)

    metrics = (
        json.loads((exp_dir / "metrics.json").read_text())
        if (exp_dir / "metrics.json").exists()
        else {}
    )
    validation = (
        json.loads((exp_dir / "validation.json").read_text())
        if (exp_dir / "validation.json").exists()
        else {}
    )
    provenance = (
        json.loads((exp_dir / "provenance.json").read_text())
        if (exp_dir / "provenance.json").exists()
        else {}
    )

    rows.append({
        "experiment_id": exp_id,
        "dataset": source_cfg["dataset"],
        "method": source_cfg["method"],
        "seed": source_cfg["seed"],
        "execution_status": status,
        "complete_artifact_set": complete,
        "scientific_qc_status": validation.get(
            "scientific_qc_status", ""
        ),
        "ARI": metric_value(metrics, "ARI"),
        "NMI": metric_value(metrics, "NMI"),
        "silhouette": metric_value(metrics, "silhouette"),
        "runtime_seconds": provenance.get("runtime_seconds"),
        "code_commit": provenance.get("code_commit", ""),
        "drive_directory": str(exp_dir),
    })

results_csv = DRIVE_OUT / "PHASE_3D_RESULTS.csv"

with results_csv.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(
        stream,
        fieldnames=list(rows[0].keys()),
    )
    writer.writeheader()
    writer.writerows(rows)


def finite_float(value):
    try:
        number = float(value)
        return number if math.isfinite(number) else None
    except Exception:
        return None


groups = defaultdict(list)

for row in rows:
    if (
        row["execution_status"] == "SUCCEEDED"
        and row["complete_artifact_set"]
    ):
        groups[(row["dataset"], row["method"])].append(row)

aggregates = []

for (dataset, method), items in sorted(groups.items()):
    record = {
        "dataset": dataset,
        "method": method,
        "n_runs": len(items),
    }

    for metric in [
        "ARI",
        "NMI",
        "silhouette",
        "runtime_seconds",
    ]:
        values = [
            finite_float(item[metric])
            for item in items
        ]
        values = [
            value for value in values
            if value is not None
        ]

        if values:
            mean = sum(values) / len(values)

            if len(values) > 1:
                sd = (
                    sum(
                        (value - mean) ** 2
                        for value in values
                    )
                    / (len(values) - 1)
                ) ** 0.5
            else:
                sd = 0.0

            record[f"{metric}_mean"] = mean
            record[f"{metric}_sd"] = sd
        else:
            record[f"{metric}_mean"] = ""
            record[f"{metric}_sd"] = ""

    aggregates.append(record)

aggregate_fields = [
    "dataset",
    "method",
    "n_runs",
    "ARI_mean",
    "ARI_sd",
    "NMI_mean",
    "NMI_sd",
    "silhouette_mean",
    "silhouette_sd",
    "runtime_seconds_mean",
    "runtime_seconds_sd",
]

aggregates_csv = DRIVE_OUT / "PHASE_3D_AGGREGATES.csv"

with aggregates_csv.open(
    "w",
    newline="",
    encoding="utf-8",
) as stream:
    writer = csv.DictWriter(
        stream,
        fieldnames=aggregate_fields,
    )
    writer.writeheader()
    writer.writerows(aggregates)

run_manifest = {
    "phase": "3D",
    "git_head": HEAD,
    "branch": BRANCH,
    "scot_commit": SCOT_COMMIT,
    "experiment_python": "3.11.8",
    "frozen_packages": FROZEN_PACKAGES,
    "expected_runs": 30,
    "succeeded_complete": sum(
        row["execution_status"] == "SUCCEEDED"
        and row["complete_artifact_set"]
        for row in rows
    ),
    "registry_policy":
        "DEFERRED_UNTIL_POST_RUN_REVIEW",
}

run_manifest_path = (
    DRIVE_OUT
    / "PHASE_3D_RUN_MANIFEST.json"
)

run_manifest_path.write_text(
    json.dumps(run_manifest, indent=2) + "\n"
)

print("\n" + "=" * 110)
print("FINAL PHASE 3D ACCOUNTING")
print("=" * 110)

for row in rows:
    print(
        f"{row['experiment_id']:<48} "
        f"{row['execution_status']:<10} "
        f"complete={str(row['complete_artifact_set']):<5} "
        f"ARI={row['ARI']} "
        f"NMI={row['NMI']} "
        f"Sil={row['silhouette']}"
    )

succeeded = sum(
    row["execution_status"] == "SUCCEEDED"
    and row["complete_artifact_set"]
    for row in rows
)

print("\nTOTAL EXPECTED:", len(rows))
print("SUCCEEDED + COMPLETE:", succeeded)
print("NOT COMPLETE:", len(rows) - succeeded)

print("\nResults CSV:", results_csv)
print("Aggregates CSV:", aggregates_csv)
print("Run manifest:", run_manifest_path)

if succeeded != 30:
    raise RuntimeError(
        f"Phase 3D incomplete: {succeeded}/30 successful complete runs"
    )

if len(aggregates) != 10:
    raise RuntimeError(
        f"Expected 10 aggregate groups, got {len(aggregates)}"
    )

if any(record["n_runs"] != 3 for record in aggregates):
    raise RuntimeError(
        "At least one aggregate group does not have all 3 seeds"
    )

print("\nPHASE 3D EXECUTION COMPLETE: 30/30")
print(
    "AGGREGATION COMPLETE: "
    "10/10 dataset-method groups, 3 seeds each"
)


When the final cell reaches `PHASE 3D EXECUTION COMPLETE: 30/30`, send back the whole commit-specific folder under `MyDrive/phase3d_results/`, or at minimum `PHASE_3D_RESULTS.csv`, `PHASE_3D_AGGREGATES.csv`, and `PHASE_3D_RUN_MANIFEST.json`.